# 01 · Carga de datos, comprobaciones y análisis exploratorio

**Semanas 1 y 2 de la planificación.** En este notebook:

1. Cargamos el dataset FMA y nos quedamos con el subconjunto *small* (8.000 canciones, 8 géneros).
2. Comprobamos que los datos están bien: cuántas canciones hay por género y partición, si hay artistas repetidos entre particiones y si faltan valores.
3. Hacemos un análisis exploratorio (EDA): miramos cómo se reparten algunas características según el género y cuánto se parecen unas características a otras (correlación).
4. Estandarizamos los datos y los guardamos ya preparados para los siguientes notebooks.

> **Antes de empezar:** descargad `fma_metadata.zip` desde el enlace que aparece en https://github.com/mdeff/fma (unos 340 MiB) y descomprimidlo dentro de la carpeta `datos/`. Al final debe existir el archivo `datos/fma_metadata/tracks.csv`.

In [ ]:
# Si trabajáis en Google Colab, quitad los '#' de estas líneas para descargar los datos allí:
# !mkdir -p ../datos && cd ../datos && wget -q https://os.unil.cloud.switch.ch/fma/fma_metadata.zip && unzip -q fma_metadata.zip

In [ ]:
import sys
sys.path.append("..")   # para poder importar nuestro código de la carpeta src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.datos import cargar_fma_small, Estandarizador, guardar_procesados

CARPETA_DATOS = "../datos/fma_metadata"
CARPETA_FIGURAS = "../figuras"


## 1. Carga de los datos

`features.csv` contiene las características de las 106.574 canciones del dataset completo, así que tarda un poco en leerse (alrededor de un minuto). La función `cargar_fma_small` (en `src/datos.py`) lo lee y se queda solo con las 8.000 canciones del subconjunto *small*.

In [ ]:
X, genero, particion, artista = cargar_fma_small(CARPETA_DATOS)

print("Canciones:", X.shape[0])
print("Características por canción:", X.shape[1])
X.iloc[:5, :6]   # primeras filas y columnas, para ver qué aspecto tienen

Cada fila es una canción y cada columna una característica. El nombre de la columna indica qué se ha medido: por ejemplo, `mfcc_mean_01` es la **media** (a lo largo de los 30 segundos) del **primer coeficiente MFCC**, que describe el timbre.

## 2. Comprobaciones básicas

### 2.1 Canciones por género y partición
Según la documentación, cada género debería tener 1.000 canciones repartidas 800 / 100 / 100 entre entrenamiento, validación y test. Lo comprobamos con una tabla de doble entrada (`crosstab`).

In [ ]:
tabla = pd.crosstab(genero, particion)[["train", "val", "test"]]
tabla["total"] = tabla.sum(axis=1)
tabla

Si las clases están equilibradas (todas con el mismo número de canciones), la **exactitud** (porcentaje de aciertos) es una buena métrica: un modelo que respondiera al azar acertaría un 1/8 = 12,5 %.

### 2.2 ¿Hay artistas repartidos entre particiones?
Si un mismo artista apareciera en entrenamiento y en test, el modelo podría aprender a reconocer su estilo personal en lugar del género, y el resultado en test sería engañosamente bueno. La partición oficial de FMA está hecha para evitarlo; lo comprobamos.

In [ ]:
artistas_train = set(artista[particion == "train"])
artistas_val = set(artista[particion == "val"])
artistas_test = set(artista[particion == "test"])

print("Artistas en train y val a la vez: ", len(artistas_train & artistas_val))
print("Artistas en train y test a la vez:", len(artistas_train & artistas_test))
print("Artistas en val y test a la vez:  ", len(artistas_val & artistas_test))

El símbolo `&` calcula la intersección de dos conjuntos (los artistas que están en ambos). Lo esperable es que salga 0 en las tres líneas.

### 2.3 Valores ausentes y características constantes

In [ ]:
print("Valores ausentes (NaN):", X.isna().sum().sum())
print("Valores infinitos:", np.isinf(X.values).sum())
print("Características constantes (desviación 0):", (X.std() == 0).sum())

Un valor ausente impediría hacer las cuentas, y una característica constante no aporta información (vale lo mismo para todas las canciones). Si alguna de las tres cifras no fuera 0, habría que decidir qué hacer con esas filas o columnas.

## 3. Análisis exploratorio

A partir de aquí **solo miramos el conjunto de entrenamiento**. El test se reserva para el final del proyecto: si lo usáramos ahora para tomar decisiones, la evaluación final ya no sería imparcial.

In [ ]:
X_train_df = X[particion == "train"]
genero_train = genero[particion == "train"]
GENEROS = sorted(genero.unique())
GENEROS

### 3.1 ¿Cambian las características según el género?

Dibujamos diagramas de caja de cuatro características fáciles de interpretar:

- `spectral_centroid_mean_01`: **brillo** del sonido (valores altos = más agudos).
- `rmse_mean_01`: **energía** media (volumen).
- `zcr_mean_01`: tasa de cruces por cero, alta en sonidos **ruidosos o percusivos**.
- `spectral_bandwidth_mean_01`: **anchura** del espectro (sonidos con muchas frecuencias a la vez).

Cada caja muestra dónde está la mitad central de las canciones de ese género, y la línea de dentro es la mediana. Si las cajas de dos géneros están a alturas distintas, esa característica ayuda a distinguirlos.

In [ ]:
caracteristicas = {
    "spectral_centroid_mean_01": "Brillo (centroide espectral)",
    "rmse_mean_01": "Energía (RMS)",
    "zcr_mean_01": "Cruces por cero",
    "spectral_bandwidth_mean_01": "Anchura espectral",
}

fig, ejes = plt.subplots(2, 2, figsize=(13, 8))
for eje, (columna, titulo) in zip(ejes.ravel(), caracteristicas.items()):
    datos_por_genero = [X_train_df.loc[genero_train == g, columna] for g in GENEROS]
    eje.boxplot(datos_por_genero, showfliers=False)   # showfliers=False: no dibuja los puntos extremos
    eje.set_xticks(range(1, len(GENEROS) + 1), GENEROS, rotation=35, ha="right")
    eje.set_title(titulo)
fig.suptitle("Distribución de cuatro características por género (entrenamiento)")
fig.tight_layout()
fig.savefig(f"{CARPETA_FIGURAS}/01_cajas_por_genero.png", dpi=150)
plt.show()

**Qué mirar en la figura:** qué géneros quedan claramente arriba o abajo en cada característica y cuáles se solapan casi por completo. Los géneros que se solapan en todo serán los que más confunda el modelo. Apuntad lo que veáis: es material para el apartado de "qué se ha hecho" de la Entrega 2.

### 3.2 ¿Se parecen mucho unas características a otras?

Muchas de las 518 características miden casi lo mismo (por ejemplo, la media y la mediana de un mismo descriptor). Lo medimos con la **correlación**: vale 1 o −1 si dos características varían a la par y 0 si no tienen relación lineal.

In [ ]:
correlaciones = X_train_df.corr().abs().values

# Nos quedamos con cada pareja una sola vez (la matriz es simétrica) y sin la diagonal
filas, columnas = np.triu_indices_from(correlaciones, k=1)
valores = correlaciones[filas, columnas]

print("Parejas de características:", len(valores))
print("Parejas con |correlación| > 0.9:", (valores > 0.9).sum())
print("Parejas con |correlación| > 0.99:", (valores > 0.99).sum())

# Las 5 parejas más correlacionadas
orden = np.argsort(valores)[::-1][:5]
nombres = X_train_df.columns
for i in orden:
    print(f"  {nombres[filas[i]]}  ~  {nombres[columnas[i]]}:  {valores[i]:.3f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5.5))

ejes[0].hist(valores, bins=50, color="steelblue")
ejes[0].set_xlabel("|correlación| entre dos características")
ejes[0].set_ylabel("Número de parejas")
ejes[0].set_title("Histograma de correlaciones")

imagen = ejes[1].imshow(correlaciones, cmap="viridis", vmin=0, vmax=1)
ejes[1].set_title("Matriz de correlaciones (518 x 518)")
fig.colorbar(imagen, ax=ejes[1], label="|correlación|")

fig.tight_layout()
fig.savefig(f"{CARPETA_FIGURAS}/01_correlaciones.png", dpi=150)
plt.show()

**Por qué nos importa:** si dos columnas de X son casi iguales, la matriz XᵀWX que usa Newton-Raphson es casi singular (casi no se puede invertir). Es el problema de "Hessiana singular" de las diapositivas, y es la razón por la que en el notebook 03 añadimos la regularización ridge. Los cuadrados brillantes de la matriz son grupos de características muy parecidas entre sí.

### 3.3 Valores atípicos

Contamos, para cada característica, cuántas canciones quedan a más de 5 desviaciones típicas de la media. Son valores muy alejados del resto.

In [ ]:
z = (X_train_df - X_train_df.mean()) / X_train_df.std()
atipicos_por_caracteristica = (z.abs() > 5).sum().sort_values(ascending=False)

print("Valores a más de 5 desviaciones:", int(atipicos_por_caracteristica.sum()),
      "de", z.size, f"({100 * atipicos_por_caracteristica.sum() / z.size:.2f} %)")
atipicos_por_caracteristica.head(8)

**Decisión:** no eliminamos estas canciones. Son canciones reales, y los valores extremos aparecen sobre todo en estadísticos como la curtosis o el máximo, que por definición reaccionan mucho a momentos puntuales de la canción. Eliminarlas supondría perder datos sin un motivo claro. Lo dejamos anotado como posible mejora (por ejemplo, transformar esas columnas) y la regularización ayudará a que no pesen demasiado.

## 4. Estandarización y guardado

Las características tienen escalas muy distintas (unas valen décimas y otras miles). Las pasamos todas a media 0 y desviación 1 por tres motivos:

- **k-NN** mide distancias: sin estandarizar, las características con valores grandes dominarían la distancia.
- **Newton-Raphson** es numéricamente más estable si todas las columnas tienen una escala parecida.
- Los **coeficientes** de la regresión logística se pueden comparar entre sí: todos indican el efecto de aumentar una desviación típica.

La media y la desviación se calculan **solo con el entrenamiento** y se aplican igual a validación y test.

In [ ]:
X_train = X[particion == "train"].values
X_val = X[particion == "val"].values
X_test = X[particion == "test"].values
y_train = genero[particion == "train"].values
y_val = genero[particion == "val"].values
y_test = genero[particion == "test"].values

estandarizador = Estandarizador().ajustar(X_train)
X_train = estandarizador.transformar(X_train)
X_val = estandarizador.transformar(X_val)
X_test = estandarizador.transformar(X_test)

print("Media de las primeras 3 columnas en train:", X_train.mean(axis=0)[:3].round(6))
print("Desviación de las primeras 3 columnas en train:", X_train.std(axis=0)[:3].round(6))
print("Media de las primeras 3 columnas en val:", X_val.mean(axis=0)[:3].round(3), "(no exactamente 0, es normal)")

In [ ]:
guardar_procesados(
    "../datos/procesados.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    nombres=np.array(X.columns), generos=np.array(GENEROS),
)
print("Datos guardados en datos/procesados.npz")

## Resumen

- Hemos cargado las 8.000 canciones del subconjunto *small* con sus 518 características.
- Hemos comprobado el reparto por género y partición, que no hay artistas compartidos entre particiones y que no faltan valores.
- El EDA muestra cómo cambian algunas características según el género y que hay muchas características muy correlacionadas, lo que justifica la regularización.
- Los datos estandarizados quedan guardados en `datos/procesados.npz`, así los demás notebooks no tienen que volver a leer los CSV.